# Airbnb listing price modeling and geospatial exploration

This notebook cleans listing-level data, engineers features, compares regression models, and visualizes price patterns. Obtain and cite the source data before running; raw listings are not included.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('data/listings.csv')
df.head()

# Data Cleaning

In [ ]:
# checking for nas 
df.isna().sum()

In [ ]:
#DROPPING UNNECESSARY COLUMNS
df.drop(columns=['listing_url', 
                 'scrape_id',
                 'neighbourhood', 
                 'bathrooms_text',
                 'last_review',
                 'last_scraped',
                 'source',
                 'description',
                 'neighbourhood_group',
                 'picture_url',
                 'host_url',
                 'host_since',
                 'host_location',
                 'host_about',
                 'host_neighbourhood',
                 'host_picture_url',
                 'host_thumbnail_url',
                 'last_review',
                 'neighbourhood_group_cleansed',
                 'host_has_profile_pic',
                 'host_verifications',
                 'host_listings_count',
                 'calendar_updated',
                 'estimated_revenue_l365d',
                 'host_profile_id',
                 'host_profile_url',
                 'first_review',
                 'hosts_time_as_user_years',
                 'hosts_time_as_user_months',
                 'hosts_time_as_host_years',
                 'minimum_minimum_nights',
                 'maximum_minimum_nights',
                 'minimum_maximum_nights',
                 'maximum_maximum_nights',
                 'hosts_time_as_host_months'],inplace=True)


# handling missing values 
df = df[df[['price']].notna().all(axis=1)]
df['host_response_time'].fillna('0', inplace=True)
df['host_response_rate'].fillna('0', inplace=True)
df['host_acceptance_rate'].fillna('0', inplace=True)
df['bedrooms'].fillna('1', inplace=True)
df['review_scores_rating'].fillna('0', inplace=True)
df['has_availability'].fillna('1', inplace=True)
df['host_is_superhost'].fillna('0', inplace=True)
df['host_acceptance_rate'].fillna('0', inplace=True)
df['host_response_rate'].fillna('0', inplace=True)
df['host_response_time'].fillna('0', inplace=True)
df['review_scores_accuracy'].fillna('0', inplace=True)
df['review_scores_cleanliness'].fillna('0', inplace=True)
df['review_scores_checkin'].fillna('0', inplace=True)
df['review_scores_communication'].fillna('0', inplace=True)
df['review_scores_location'].fillna('0', inplace=True)
df['review_scores_value'].fillna('0', inplace=True)
df['reviews_per_month'].fillna('0', inplace=True)
df['host_total_listings_count'].fillna('1', inplace=True)
df['host_identity_verified'].fillna('0', inplace=True)
df['license'].fillna('0', inplace=True)
df["host_name"].fillna("Unknown", inplace=True)
cleaneddf = df.drop(columns=['neighborhood_overview','neighbourhood_cleansed'])
cleaneddf = cleaneddf.dropna()
cleaneddf.isna().sum()

In [ ]:
#ensuring correct data types for modeling
cleaneddf["price"] = cleaneddf["price"].str.replace(r'[$,]', '', regex=True).astype(float)


cleaneddf["latitude"] = pd.to_numeric(cleaneddf["latitude"], errors="coerce")
cleaneddf["longitude"] = pd.to_numeric(cleaneddf["longitude"], errors="coerce")
cleaneddf["minimum_nights"] = pd.to_numeric(cleaneddf["minimum_nights"], errors="coerce")
cleaneddf["number_of_reviews"] = pd.to_numeric(cleaneddf["number_of_reviews"], errors="coerce")
cleaneddf["minimum_nights"] = pd.to_numeric(cleaneddf["minimum_nights"], errors="coerce")
cleaneddf["reviews_per_month"] = pd.to_numeric(cleaneddf["reviews_per_month"], errors="coerce")
cleaneddf["calculated_host_listings_count"] = pd.to_numeric(cleaneddf["calculated_host_listings_count"], errors="coerce")
cleaneddf["availability_365"] = pd.to_numeric(cleaneddf["availability_365"], errors="coerce")

cleaneddf["host_response_rate"] = cleaneddf["host_response_rate"].str.replace('%', '', regex=True)
cleaneddf["host_response_rate"] = pd.to_numeric(cleaneddf["host_response_rate"], errors="coerce")
cleaneddf["host_acceptance_rate"] = cleaneddf["host_acceptance_rate"].str.replace('%', '', regex=True)
cleaneddf["host_acceptance_rate"] = pd.to_numeric(cleaneddf["host_acceptance_rate"], errors="coerce")


cleaneddf["host_total_listings_count"] = pd.to_numeric(cleaneddf["host_total_listings_count"], errors="coerce")
cleaneddf["accommodates"] = pd.to_numeric(cleaneddf["accommodates"], errors="coerce")
cleaneddf["bedrooms"] = pd.to_numeric(cleaneddf["bedrooms"], errors="coerce")
cleaneddf["review_scores_rating"] = pd.to_numeric(cleaneddf["review_scores_rating"], errors="coerce")
cleaneddf["review_scores_accuracy"] = pd.to_numeric(cleaneddf["review_scores_accuracy"], errors="coerce")
cleaneddf["review_scores_cleanliness"] = pd.to_numeric(cleaneddf["review_scores_cleanliness"], errors="coerce")
cleaneddf["review_scores_checkin"] = pd.to_numeric(cleaneddf["review_scores_checkin"], errors="coerce")
cleaneddf["review_scores_communication"] = pd.to_numeric(cleaneddf["review_scores_communication"], errors="coerce")
cleaneddf["review_scores_location"] = pd.to_numeric(cleaneddf["review_scores_location"], errors="coerce")
cleaneddf["review_scores_value"] = pd.to_numeric(cleaneddf["review_scores_value"], errors="coerce")
cleaneddf["review_scores_checkin"] = pd.to_numeric(cleaneddf["review_scores_checkin"], errors="coerce")


# Feature Engineering

In [ ]:

df = cleaneddf.copy()

# ---------------------------
# 1. License: anything other than 0 becomes 1
# ---------------------------
df["license"] = df["license"].astype(str).str.strip()
df["license"] = np.where(df["license"] == "0", 0, 1)

# ---------------------------
# 2. Binary encode t/f columns
# ---------------------------
binary_tf_cols = [
    "host_is_superhost",
    "host_identity_verified",
    "has_availability",
    "instant_bookable"
]

for col in binary_tf_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"t": 1, "f": 0, "true": 1, "false": 0})
    )

# fill anything unmapped
for col in binary_tf_cols:
    df[col] = df[col].fillna(0).astype(int)

# ---------------------------
# 3. One-hot encode categorical columns
# ---------------------------
categorical_cols = [
    "host_response_time",
    "property_type",
    "room_type",
    'city'
]

for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)

# ---------------------------
# 4. Keyword extraction from amenities
# ---------------------------
df["amenities"] = df["amenities"].fillna("").astype(str).str.lower()

amenity_patterns = {
    "wifi_am": r"wifi|wi-fi|internet",
    "ac_am": r"air conditioning|ac\b",
    "kitchen_am": r"\bkitchen\b",
    "washer_am": r"\bwasher\b",
    "dryer_am": r"\bdryer\b",
    "laundry_am": r"laundry",
    "parking_am": r"free parking|paid parking|parking|garage",
    "pool_am": r"\bpool\b",
    "hot_tub_am": r"hot tub|jacuzzi|spa",
    "gym_am": r"\bgym\b|fitness",
    "elevator_am": r"elevator|lift",
    "pets_allowed_am": r"pets allowed|pet friendly",
    "tv_am": r"\btv\b|television",
    "workspace_am": r"dedicated workspace|workspace|desk",
    "balcony_am": r"balcony|patio|terrace|deck",
    "fireplace_am": r"fireplace",
    "bbq_am": r"bbq|grill",
    "beach_access_am": r"beach access",
    "waterfront_am": r"waterfront|lake access|river access",
    "smoke_alarm_am": r"smoke alarm",
    "carbon_monoxide_alarm_am": r"carbon monoxide alarm",
    "self_checkin_am": r"self check-in|smart lock|keypad|lockbox",
    "breakfast_am": r"breakfast",
    "crib_am": r"\bcrib\b",
    "hair_dryer_am": r"hair dryer",
    "iron_am": r"\biron\b"
}

for col, pattern in amenity_patterns.items():
    df[col] = df["amenities"].str.contains(pattern, regex=True, na=False).astype(int)

# Analyzing name of airbnb to create binary coplums of keywords in the name of the listing.

property_patterns = {
    "studio": r"\bstudio\b",
    "apartment": r"\bapt\b|\bapartment\b",
    "condo": r"\bcondo\b",
    "house": r"\bhouse\b|\bhome\b",
    "loft": r"\bloft\b",
    "suite": r"\bsuite\b",
    "townhouse": r"\btownhouse\b",
    "brownstone": r"\bbrownstone\b",
    "duplex": r"\bduplex\b",
    "triplex": r"\btriplex\b",
    "room": r"\broom\b",
    "private_room": r"private room",
    "shared_room": r"shared room",
    "entire_place": r"entire place|entire home|entire apt"
}

size_patterns = {
    "1br": r"1 ?br|1 bedroom",
    "2br": r"2 ?br|2 bedroom",
    "3br": r"3 ?br|3 bedroom",
    "4plus_br": r"4 ?br|5 ?br|6 ?br",
    "bath": r"bath|ba\b",
    "large": r"large|big|spacious",
    "small": r"small|tiny|compact",
    "sleeps": r"sleeps \d+"
}

transport = {
    "subway": r"subway|metro|t stop|t line",
    "red_line": r"red line",
    "green_line": r"green line",
    "orange_line": r"orange line",
    "blue_line": r"blue line",
    "walk": r"walk to|steps to|minutes to",
    "downtown": r"downtown|city center",
    "airport": r"airport|logan",
    "commute": r"commute|easy access"
}

amenities = {
    "parking": r"parking|garage",
    "ac": r"\bac\b|air ?conditioning",
    "wifi": r"wifi|wi-fi",
    "kitchen": r"kitchen",
    "deck": r"deck|balcony|patio|terrace",
    "garden": r"garden",
    "private_entrance": r"private entrance",
    "washer_dryer": r"washer|dryer|laundry",
    "workspace": r"workspace|desk|office",
    "pet_friendly": r"pet|dog|pets ok",
    "elevator": r"elevator"
}


views = {
    "view": r"\bview\b|views",
    "river_view": r"river|water|harbor",
    "park_view": r"park",
    "city_view": r"city|skyline",
    "quiet": r"quiet|peaceful",
    "safe": r"safe",
    "sunny": r"sunny|bright|light-filled"
}

style = {
    "luxury": r"luxury|luxurious|premium|high-end",
    "modern": r"modern|updated|renovated|new",
    "clean": r"clean|spotless",  
    "cozy": r"cozy|charming|warm",
    "spacious": r"spacious|open",
    "stylish": r"stylish|chic|elegant",
    "historic": r"historic|victorian",
    "trendy": r"trendy|hip"
}

audience = {
    "family": r"family|kids",
    "business": r"business|corporate|executive",
    "romantic": r"romantic",
    "travelers": r"travelers|tourists",
    "extended_stay": r"extended stay|long term"
}

negative = {
    "budget": r"budget|cheap|affordable",
    "basic": r"basic|simple",
    "shared": r"shared",
    "room_only": r"room",
}


all_patterns = {**property_patterns, **size_patterns, 
                **transport, **amenities, **views, **style,
                **audience, **negative}

for col, pattern in all_patterns.items():
    cleaneddf[col] = cleaneddf["name"].str.contains(pattern, case=False, na=False).astype(int)


# ---------------------------
# 7. Final check
# ---------------------------
print(df.shape)
print(df.dtypes.head(20))

In [ ]:
#handling outliers 

from scipy import stats
import numpy as np
z = np.abs(stats.zscore(df['price']))
zThreshold = 2.5

mask = z <= zThreshold
zOutlierDF = df[mask]



print("Original Shape:", df.shape)
print("After Removing Outliers:", zOutlierDF.shape)

df = zOutlierDF.copy()

# Visualizations


In [ ]:
import matplotlib.pyplot as plt

hist_cols = [
    "price", "minimum_nights", "maximum_nights",
    "number_of_reviews", "reviews_per_month",
    "availability_365", "availability_30",
    "estimated_occupancy_l365d",
    "review_scores_rating", "review_scores_cleanliness",
    "review_scores_location", "review_scores_value"
]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(hist_cols):
    data = df[col].dropna()
    
    # trim outliers
    data = data[data < data.quantile(0.99)]
    
    axes[i].hist(data, bins=30)
    axes[i].set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
bar_cols = [
    "host_is_superhost",
    "host_identity_verified",
    "instant_bookable",
    "has_availability"
]

binary_means = df[bar_cols].mean()

binary_means.plot(kind="bar")

plt.title("Binary Feature Distribution (Proportion = 1)")
plt.ylabel("Proportion")
plt.xticks(rotation=45)
plt.show()

In [ ]:
host_response_time_cols = [
    "host_response_time_a few days or more",
    "host_response_time_within a day",
    "host_response_time_within a few hours",
    "host_response_time_within an hour"
]

response_means = df[host_response_time_cols].mean()

response_means.index = response_means.index.str.replace("host_response_time_", "")

response_means.plot(kind="bar")

plt.title("Host Response Time Distribution")
plt.ylabel("Proportion")
plt.xticks(rotation=30)
plt.show()

In [ ]:
room_type_cols = [
    "room_type_Hotel room",
    "room_type_Private room",
    "room_type_Shared room"
]

room_means = df[room_type_cols].mean()

room_means.index = room_means.index.str.replace("room_type_", "")

room_means.plot(kind="bar")

plt.title("Room Type Distribution")
plt.ylabel("Proportion")
plt.show()

In [ ]:
property_cols = [col for col in df.columns if col.startswith("property_type_")]

property_means = df[property_cols].mean().sort_values(ascending=False).head(10)

property_means.index = property_means.index.str.replace("property_type_", "").str.replace("_", " ")

property_means.plot(kind="barh")

plt.title("Top Property Types")
plt.xlabel("Proportion")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
city_cols = [col for col in df.columns if col.startswith("city_")]

city_means = df[city_cols].mean().sort_values(ascending=False)

city_means.index = city_means.index.str.replace("city_", "").str.replace("_", " ")

city_means.head(10).plot(kind="bar")

plt.title("Top Cities/Counties by Listings")
plt.ylabel("Proportion")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# correlation analysis

import pandas as pd

# Select only numeric columns
numeric_df = df.select_dtypes(include=["number"])

# Compute correlations with price
correlations = numeric_df.corr()["price"].drop("price")

# Get top 10 strongest correlations (absolute value)
top10 = correlations.abs().sort_values(ascending=False).head(10)

# Get actual values (with sign)
top10_with_sign = correlations[top10.index]

# Modeling

In [ ]:
import numpy as np

df_model = df.copy()

# Target
yDF = df_model["price"]

# Drop non-usable columns
drop_cols = ['market', 'state','geo_type', 'amenities', "calendar_last_scraped", "id", "name", "host_id", "host_name", "latitude", "longitude", "price"]
xDF = df_model.drop(columns=drop_cols, errors="ignore")



In [ ]:
xDF

In [ ]:
from sklearn.model_selection import train_test_split

xTrain, xTest, yTrain, yTest = train_test_split(
    xDF, yDF, 
    test_size=0.05,
    shuffle = True)

from sklearn.preprocessing import StandardScaler

xScaler = StandardScaler()
xColNames = xTrain.columns.values.tolist()
# train the scaler and apply it to the training set
xTrainScaled = xScaler.fit_transform(xTrain[xColNames])

In [ ]:
## MODEL 1 Grid Search for Linear Regression ##
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

from sklearn.model_selection import GridSearchCV

from sklearn.linear_model import LinearRegression


parameters = {
              "fit_intercept": [True, False],
              "positive" : [True, False]
             }
gridSearch = GridSearchCV(estimator = LinearRegression(),  # the model
                          param_grid = parameters, # the hyperparameters to tune
                          scoring = 'r2', # the metric to optimize
                          cv = 5, # number of folds for cross-validation
                          verbose=0) 


gridSearch.fit(xTrainScaled, yTrain) 

linearParams = gridSearch.best_params_

linearParams

In [ ]:
## MODEL 2 Grid Search for Lasso Regression ##
from sklearn.linear_model import Lasso

parameters = {
              "alpha": [0.001, 0.01, 0.1, 1, 10],
              "max_iter": [5000, 10000]
             }

gridSearch = GridSearchCV(estimator = Lasso(),
                          param_grid = parameters,
                          scoring = 'r2',
                          cv = 5,
                          verbose=0)

gridSearch.fit(xTrainScaled, yTrain)

lassoParams = gridSearch.best_params_

lassoParams

In [ ]:
## MODEL 3 Grid Search for Decision Tree Regression ##
from sklearn.tree import DecisionTreeRegressor

parameters = {
              "max_depth": [5, 10, 15, 20],
              "min_samples_split": [2, 10, 20],
              "min_samples_leaf": [1, 5, 10]
             }

gridSearch = GridSearchCV(estimator = DecisionTreeRegressor(random_state=42),
                          param_grid = parameters,
                          scoring = 'r2',
                          cv = 5,
                          verbose=0)

gridSearch.fit(xTrainScaled, yTrain)

dtParams = gridSearch.best_params_

dtParams

In [ ]:
## MODEL 4 Grid Search for Random Forest Regression ##
from sklearn.ensemble import RandomForestRegressor

parameters = {
              "n_estimators": [100, 200],
              "max_depth": [10, 20, None],
              "min_samples_split": [2, 10],
              "min_samples_leaf": [1, 5]
             }

gridSearch = GridSearchCV(estimator = RandomForestRegressor(random_state=42),
                          param_grid = parameters,
                          scoring = 'r2',
                          cv = 3,
                          verbose=0)

gridSearch.fit(xTrainScaled, yTrain)

rfParams = gridSearch.best_params_

rfParams

In [ ]:
## MODEL 5 Grid Search for Gradient Boosting Regression ##
from sklearn.ensemble import GradientBoostingRegressor

parameters = {
              "n_estimators": [100, 200],
              "learning_rate": [0.05, 0.1],
              "max_depth": [3, 5]
             }

gridSearch = GridSearchCV(estimator = GradientBoostingRegressor(random_state=42),
                          param_grid = parameters,
                          scoring = 'r2',
                          cv = 3,
                          verbose=0)

gridSearch.fit(xTrainScaled, yTrain)

gbParams = gridSearch.best_params_

gbParams

In [ ]:
## MODEL 6 Grid Search for KNN Regression ##
from sklearn.neighbors import KNeighborsRegressor

parameters = {
              "n_neighbors": [3, 5, 10, 15],
              "weights": ["uniform", "distance"],
              "p": [1, 2]
             }

gridSearch = GridSearchCV(estimator = KNeighborsRegressor(),
                          param_grid = parameters,
                          scoring = 'r2',
                          cv = 5,
                          verbose=0)

gridSearch.fit(xTrainScaled, yTrain)

knnParams = gridSearch.best_params_

knnParams

In [ ]:
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
## MODEL 1 LINEAR REGRESSION WITH K-FOLD CROSS-VALIDATION ##

rmseScores = []
r2Scores = []

for i, (trainIndex, valIndex) in enumerate(cv.split(xTrain, yTrain)):
    xLocalTrain, xVal = xTrain.iloc[trainIndex], xTrain.iloc[valIndex]
    yLocalTrain, yVal = yTrain.iloc[trainIndex], yTrain.iloc[valIndex]

    xScaler = StandardScaler()
    xTrainScaledLocal = xScaler.fit_transform(xLocalTrain)
    xValScaled = xScaler.transform(xVal)

    clf = LinearRegression(**linearParams)
    clf.fit(xTrainScaledLocal, yLocalTrain)

    yPred = clf.predict(xValScaled)

    rmseScores.append(root_mean_squared_error(yVal, yPred))
    r2Scores.append(r2_score(yVal, yPred))

    print(f"Completed Fold {i}")
    print(f"    RMSE={rmseScores[i]}    R^2={r2Scores[i]}")

linearMeanRMSE = sum(rmseScores) / len(rmseScores)
linearMeanR2 = sum(r2Scores) / len(r2Scores)

print("Mean RMSE score:", linearMeanRMSE)
print("Mean R^2 score:", linearMeanR2)

In [ ]:
## MODEL 2 LASSO REGRESSION WITH K-FOLD CROSS-VALIDATION ##

rmseScores = []
r2Scores = []

for i, (trainIndex, valIndex) in enumerate(cv.split(xTrain, yTrain)):
    xLocalTrain, xVal = xTrain.iloc[trainIndex], xTrain.iloc[valIndex]
    yLocalTrain, yVal = yTrain.iloc[trainIndex], yTrain.iloc[valIndex]

    xScaler = StandardScaler()
    xTrainScaledLocal = xScaler.fit_transform(xLocalTrain)
    xValScaled = xScaler.transform(xVal)

    clf = Lasso(**lassoParams)
    clf.fit(xTrainScaledLocal, yLocalTrain)

    yPred = clf.predict(xValScaled)

    rmseScores.append(root_mean_squared_error(yVal, yPred))
    r2Scores.append(r2_score(yVal, yPred))

    print(f"Completed Fold {i}")
    print(f"    RMSE={rmseScores[i]}    R^2={r2Scores[i]}")

lassoMeanRMSE = sum(rmseScores) / len(rmseScores)
lassoMeanR2 = sum(r2Scores) / len(r2Scores)

print("Mean RMSE score:", lassoMeanRMSE)
print("Mean R^2 score:", lassoMeanR2)

In [ ]:
## MODEL 3 DECISION TREE WITH K-FOLD CROSS-VALIDATION ##

rmseScores = []
r2Scores = []

for i, (trainIndex, valIndex) in enumerate(cv.split(xTrain, yTrain)):
    xLocalTrain, xVal = xTrain.iloc[trainIndex], xTrain.iloc[valIndex]
    yLocalTrain, yVal = yTrain.iloc[trainIndex], yTrain.iloc[valIndex]

    clf = DecisionTreeRegressor(**dtParams, random_state=42)
    clf.fit(xLocalTrain, yLocalTrain)

    yPred = clf.predict(xVal)

    rmseScores.append(root_mean_squared_error(yVal, yPred))
    r2Scores.append(r2_score(yVal, yPred))

    print(f"Completed Fold {i}")
    print(f"    RMSE={rmseScores[i]}    R^2={r2Scores[i]}")

dtMeanRMSE = sum(rmseScores) / len(rmseScores)
dtMeanR2 = sum(r2Scores) / len(r2Scores)

print("Mean RMSE score:", dtMeanRMSE)
print("Mean R^2 score:", dtMeanR2)

In [ ]:
## MODEL 4 RANDOM FOREST WITH K-FOLD CROSS-VALIDATION ##

rmseScores = []
r2Scores = []

for i, (trainIndex, valIndex) in enumerate(cv.split(xTrain, yTrain)):
    xLocalTrain, xVal = xTrain.iloc[trainIndex], xTrain.iloc[valIndex]
    yLocalTrain, yVal = yTrain.iloc[trainIndex], yTrain.iloc[valIndex]

    clf = RandomForestRegressor(**rfParams, random_state=42)
    clf.fit(xLocalTrain, yLocalTrain)

    yPred = clf.predict(xVal)

    rmseScores.append(root_mean_squared_error(yVal, yPred))
    r2Scores.append(r2_score(yVal, yPred))

    print(f"Completed Fold {i}")
    print(f"    RMSE={rmseScores[i]}    R^2={r2Scores[i]}")

rfMeanRMSE = sum(rmseScores) / len(rmseScores)
rfMeanR2 = sum(r2Scores) / len(r2Scores)

print("Mean RMSE score:", rfMeanRMSE)
print("Mean R^2 score:", rfMeanR2)

In [ ]:
## MODEL 5 GRADIENT BOOSTING WITH K-FOLD CROSS-VALIDATION ##

rmseScores = []
r2Scores = []

for i, (trainIndex, valIndex) in enumerate(cv.split(xTrain, yTrain)):
    xLocalTrain, xVal = xTrain.iloc[trainIndex], xTrain.iloc[valIndex]
    yLocalTrain, yVal = yTrain.iloc[trainIndex], yTrain.iloc[valIndex]

    clf = GradientBoostingRegressor(**gbParams, random_state=42)
    clf.fit(xLocalTrain, yLocalTrain)

    yPred = clf.predict(xVal)

    rmseScores.append(root_mean_squared_error(yVal, yPred))
    r2Scores.append(r2_score(yVal, yPred))

    print(f"Completed Fold {i}")
    print(f"    RMSE={rmseScores[i]}    R^2={r2Scores[i]}")

gbMeanRMSE = sum(rmseScores) / len(rmseScores)
gbMeanR2 = sum(r2Scores) / len(r2Scores)

print("Mean RMSE score:", gbMeanRMSE)
print("Mean R^2 score:", gbMeanR2)

In [ ]:
## MODEL 6 KNN WITH K-FOLD CROSS-VALIDATION ##

rmseScores = []
r2Scores = []

for i, (trainIndex, valIndex) in enumerate(cv.split(xTrain, yTrain)):
    xLocalTrain, xVal = xTrain.iloc[trainIndex], xTrain.iloc[valIndex]
    yLocalTrain, yVal = yTrain.iloc[trainIndex], yTrain.iloc[valIndex]

    xScaler = StandardScaler()
    xTrainScaledLocal = xScaler.fit_transform(xLocalTrain)
    xValScaled = xScaler.transform(xVal)

    clf = KNeighborsRegressor(**knnParams)
    clf.fit(xTrainScaledLocal, yLocalTrain)

    yPred = clf.predict(xValScaled)

    rmseScores.append(root_mean_squared_error(yVal, yPred))
    r2Scores.append(r2_score(yVal, yPred))

    print(f"Completed Fold {i}")
    print(f"    RMSE={rmseScores[i]}    R^2={r2Scores[i]}")

knnMeanRMSE = sum(rmseScores) / len(rmseScores)
knnMeanR2 = sum(r2Scores) / len(r2Scores)

print("Mean RMSE score:", knnMeanRMSE)
print("Mean R^2 score:", knnMeanR2)

In [ ]:
### feature scaling
xScaler = StandardScaler()
xColNames = xTrain.columns.values.tolist()
# train the scaler and apply it to the training and test sets
xTrainScaled = xScaler.fit_transform(xTrain[xColNames])
xTestScaled = xScaler.transform(xTest[xColNames])

In [ ]:
################ 
# Linear Regression
################
clf = LinearRegression(**linearParams)
clf.fit(xTrainScaled, yTrain)

preds = clf.predict(xTestScaled)

linearRsquared = r2_score(yTest, preds)
linearRmse = root_mean_squared_error(yTest, preds)


################ 
# Lasso Regression
################
clf = Lasso(**lassoParams)
clf.fit(xTrainScaled, yTrain)

preds = clf.predict(xTestScaled)

lassoRsquared = r2_score(yTest, preds)
lassoRmse = root_mean_squared_error(yTest, preds)


################ 
# Decision Tree Regression
################
clf = DecisionTreeRegressor(**dtParams, random_state=42)
clf.fit(xTrain, yTrain)

preds = clf.predict(xTest)

dtRsquared = r2_score(yTest, preds)
dtRmse = root_mean_squared_error(yTest, preds)


################ 
# Random Forest Regression
################
clf = RandomForestRegressor(**rfParams, random_state=42)
clf.fit(xTrain, yTrain)

preds = clf.predict(xTest)

rfRsquared = r2_score(yTest, preds)
rfRmse = root_mean_squared_error(yTest, preds)


################ 
# Gradient Boosting Regression
################
clf = GradientBoostingRegressor(**gbParams, random_state=42)
clf.fit(xTrain, yTrain)

preds = clf.predict(xTest)

gbRsquared = r2_score(yTest, preds)
gbRmse = root_mean_squared_error(yTest, preds)


################ 
# KNN Regression
################
clf = KNeighborsRegressor(**knnParams)
clf.fit(xTrainScaled, yTrain)

preds = clf.predict(xTestScaled)

knnRsquared = r2_score(yTest, preds)
knnRmse = root_mean_squared_error(yTest, preds)


################ 
# Results
################
scoreDict = {
    "model": [
        "Linear Regression",
        "Lasso Regression",
        "Decision Tree Regression",
        "Random Forest Regression",
        "Gradient Boosting Regression",
        "KNN Regression"
    ],
    "r2": [
        linearRsquared,
        lassoRsquared,
        dtRsquared,
        rfRsquared,
        gbRsquared,
        knnRsquared
    ],
    "rmse": [
        linearRmse,
        lassoRmse,
        dtRmse,
        rfRmse,
        gbRmse,
        knnRmse
    ]
}

resultsDF = pd.DataFrame.from_dict(scoreDict)

resultsDF = resultsDF.sort_values("r2", ascending=False)

In [ ]:
resultsDF

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Feature names from your training data
feature_names = xTrain.columns if hasattr(xTrain, "columns") else df.drop(columns=["price"]).columns

def plot_feature_importance(model, model_name, top_n=15):
    importances = model.feature_importances_

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    }).sort_values("Importance", ascending=False).head(top_n)

    print(f"\nTop {top_n} Feature Importances for {model_name}")
    print(importance_df)

    plt.figure(figsize=(10, 6))
    plt.barh(importance_df["Feature"][::-1], importance_df["Importance"][::-1])
    plt.title(f"Top {top_n} Feature Importances: {model_name}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

    return importance_df

In [ ]:
dt = DecisionTreeRegressor(**dtParams, random_state=42)
dt.fit(xTrain, yTrain)

dt_importance = plot_feature_importance(dt, "Decision Tree")


In [ ]:
rf = RandomForestRegressor(**rfParams, random_state=42)
rf.fit(xTrain, yTrain)

rf_importance = plot_feature_importance(rf, "Random Forest")

In [ ]:
gb = GradientBoostingRegressor(**gbParams, random_state=42)
gb.fit(xTrain, yTrain)

gb_importance = plot_feature_importance(gb, "Gradient Boosting")

In [ ]:
################ 
# Lasso Regression
################
lasso = Lasso(**lassoParams)
lasso.fit(xTrainScaled, yTrain)



################ 
# Decision Tree Regression
################
lr = LinearRegression(**linearParams)
lr.fit(xTrain, yTrain)




coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": lasso.coef_
})

coef_df["Abs_Coefficient"] = coef_df["Coefficient"].abs()

top_lasso = coef_df.sort_values("Abs_Coefficient", ascending=False).head(15)

print(top_lasso)

plt.figure(figsize=(10, 6))
plt.barh(top_lasso["Feature"][::-1], top_lasso["Coefficient"][::-1])
plt.title("Top 15 Lasso Coefficients")
plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()



coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": lr.coef_
})

coef_df["Abs_Coefficient"] = coef_df["Coefficient"].abs()

top_lr = coef_df.sort_values("Abs_Coefficient", ascending=False).head(15)

print(top_lr)

plt.figure(figsize=(10, 6))
plt.barh(top_lr["Feature"][::-1], top_lr["Coefficient"][::-1])
plt.title("Top 15 Linear Regression Coefficients")
plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()